In [1]:
import sys
import os

folder = os.path.abspath("day-04-cv-tuning/")
sys.path.insert(0, folder)

import solution4 
from solution4 import Datasets, StratifiedKfoldCV, GridSearchCV, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import sklearn.metrics as metrics
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict, Counter

prop_cycle = plt.rcParams['axes.prop_cycle']
colors = prop_cycle.by_key()['color']

# Models and setup
- StandardScaler and OneHotEncoder

In [2]:

data = Datasets('custom',split_method='sklearn')
data.load_custom_data(file_path='../data/')  

def log_reg_pipeline(case: Datasets):
    log_reg = LogisticRegression(max_iter=2000)
    param_grid = {
        'C': [1,2,3,4,5,6,7,8,9,10,12,13,14,15,16,16.5,17],
        'penalty': ['l1', 'l2'],
        'solver': ['liblinear']  # 'liblinear' supports both l1 and l2 penalties
    }
    
    
    
    
    X_train, y_train = case.X_train, case.y_train
    X_test, y_test = case.X_test, case.y_test
    
    log_reg.fit(X_train, y_train)
    
    cv = StratifiedKfoldCV(n_samples=len(X_train), k=5, seed=42)
    
    # Using cross_val_score to evaluate the model with cross-validation
    
    # Using cross_val with grid search to find the best hyperparameters
    grid_search = GridSearchCV(log_reg, param_grid, cv=cv)
    grid_search.fit(X_train, y_train)
    
    model_params = grid_search.best_params_
    model = LogisticRegression(**model_params)
    return model, model_params, cv, X_train, y_train, X_test, y_test
    
    
def decision_tree_pipeline(case: Datasets):
    dtc = DecisionTreeClassifier()
    param_grid = {
        "max_depth": [3, 5, 10, None],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [1, 2, 4]
    }
    
    X_train, y_train = case.X_train, case.y_train
    X_test, y_test = case.X_test, case.y_test
    
    dtc.fit(X_train, y_train)
    
    cv = StratifiedKfoldCV(n_samples=len(X_train), k=5, seed=42)
    
    # Using cross_val_score to evaluate the model with cross-validation
    
    # Using cross_val with grid search to find the best hyperparameters
    grid_search = GridSearchCV(dtc, param_grid, cv=cv)
    grid_search.fit(X_train, y_train)
    
    model_params = grid_search.best_params_
    model = DecisionTreeClassifier(**model_params)
    return model, model_params, cv, X_train, y_train, X_test, y_test

def knn_pipeline(case: Datasets):
    knc = KNeighborsClassifier()
    param_grid = {
        "n_neighbors": [3, 5, 7, 9],
        "weights": ["uniform", "distance"],
        "metric": ["euclidean", "manhattan"]
    }
    
    X_train, y_train = case.X_train, case.y_train
    X_test, y_test = case.X_test, case.y_test
    
    knc.fit(X_train, y_train)
    
    cv = StratifiedKfoldCV(n_samples=len(X_train), k=5, seed=42)
    
    # Using cross_val_score to evaluate the model with cross-validation
    
    # Using cross_val with grid search to find the best hyperparameters
    grid_search = GridSearchCV(knc, param_grid, cv=cv)
    grid_search.fit(X_train, y_train)
    
    model_params = grid_search.best_params_
    model = KNeighborsClassifier(**model_params)
    return model, model_params, cv, X_train, y_train, X_test, y_test
    

model, model_params, cv, X_train_dtc, y_train_dtc, X_test_dtc, y_test_dtc = decision_tree_pipeline(data)
knn, model_params, cv, X_train_knn, y_train_knn, X_test_knn, y_test_knn = knn_pipeline(data)
log_reg, model_params, cv, X_train_log, y_train_log, X_test_log, y_test_log = log_reg_pipeline(data)

In [3]:
cv_means_dct, cv_stds_dct, cv_scores_dct = cross_val_score(model, X_train_dtc, y_train_dtc, cv=cv)
cv_means_knn, cv_stds_knn, cv_scores_knn = cross_val_score(knn, X_train_knn, y_train_knn, cv=cv)
cv_means_log, cv_stds_log, cv_scores_log = cross_val_score(log_reg, X_train_log, y_train_log, cv=cv)


print("Mean scores with standard deviation For DecisionTreeClassifier:")
print("Accuracy: {:.4f} ± {:.4f}".format(cv_means_dct['accuracy'], cv_stds_dct['accuracy']))
print("Precision: {:.4f} ± {:.4f}".format(cv_means_dct['precision'], cv_stds_dct['precision']))
print("Recall: {:.4f} ± {:.4f}".format(cv_means_dct['recall'], cv_stds_dct['recall']))
print("F1 Score: {:.4f} ± {:.4f}".format(cv_means_dct['f1'], cv_stds_dct['f1']))
print("ROC-AUC: {:.4f} ± {:.4f}".format(cv_means_dct['roc_auc'], cv_stds_dct['roc_auc']))

print('-' * 30)
print("Mean scores with standard deviation for KNeighbours:")
print("Accuracy: {:.4f} ± {:.4f}".format(cv_means_knn['accuracy'], cv_stds_knn['accuracy']))
print("Precision: {:.4f} ± {:.4f}".format(cv_means_knn['precision'], cv_stds_knn['precision']))
print("Recall: {:.4f} ± {:.4f}".format(cv_means_knn['recall'], cv_stds_knn['recall']))
print("F1 Score: {:.4f} ± {:.4f}".format(cv_means_knn['f1'], cv_stds_knn['f1']))
print("ROC-AUC: {:.4f} ± {:.4f}".format(cv_means_knn['roc_auc'], cv_stds_knn['roc_auc']))
print('-' * 30)
print("Mean scores with standard deviation for LogisticRegression:")
print("Accuracy: {:.4f} ± {:.4f}".format(cv_means_log['accuracy'], cv_stds_log['accuracy']))
print("Precision: {:.4f} ± {:.4f}".format(cv_means_log['precision'], cv_stds_log['precision']))
print("Recall: {:.4f} ± {:.4f}".format(cv_means_log['recall'], cv_stds_log['recall']))
print("F1 Score: {:.4f} ± {:.4f}".format(cv_means_log['f1'], cv_stds_log['f1']))
print("ROC-AUC: {:.4f} ± {:.4f}".format(cv_means_log['roc_auc'], cv_stds_log['roc_auc']))


Mean scores with standard deviation For DecisionTreeClassifier:
Accuracy: 0.8252 ± 0.0023
Precision: 0.7297 ± 0.0085
Recall: 0.4283 ± 0.0122
F1 Score: 0.5396 ± 0.0096
ROC-AUC: 0.8726 ± 0.0027
------------------------------
Mean scores with standard deviation for KNeighbours:
Accuracy: 0.8215 ± 0.0040
Precision: 0.6513 ± 0.0108
Recall: 0.5469 ± 0.0099
F1 Score: 0.5945 ± 0.0092
ROC-AUC: 0.8531 ± 0.0031
------------------------------
Mean scores with standard deviation for LogisticRegression:
Accuracy: 0.8261 ± 0.0020
Precision: 0.6781 ± 0.0049
Recall: 0.5202 ± 0.0109
F1 Score: 0.5886 ± 0.0073
ROC-AUC: 0.8778 ± 0.0033


# Data setup

### Training dataset
The training dataset is already split, and it's stratisfied on y. This dataset is split again in the cross_validation and has been used in the KFold splittings. 


## Day 3
**imputer**
- Made a filling function to handle missing values for either num or cat columns


**numerical columns**
- Used standard scaling method like sklearn's StandardScaler

**cat columns**
- Imputed by using the most frequent variable 
- Used onehotencoder to encode categorical values.

**pre processor**
- Created a pipeline class like sklearn's method handling steps
- "Implemented" a column transformer. It doesn't work exactly like sklearn because I haven't implemented it to produce output of matrix value. This might be a bug, but doesn't matter if i don't use it in a model. 



### Debugging assisted by AI: 
- The code contains the 4 modifications done by AI. 


# Day 4

1. Implemented the k_fold_split method 
Used some code from sklearn's documentation as well as other sources listed in the script comments

2. cross_val_score 
This needs to be stripped of my testing implementations later, but it works as expected. 

3. StratisfiedKfoldCV
Same as Kfold

sources:
> https://www.geeksforgeeks.org/machine-learning/cross-validation-using-k-fold-with-scikit-learn/
> https://github.com/scikit-learn/scikit-learn/blob/cc50648cc1b759b53a4edbce0f3bb6c237349448/sklearn/model_selection/_split.py#L184
> https://medium.com/@pacosun/stratified-k-fold-cross-validation-when-balance-matters-c28b9a7cb9bc
> https://towardsdatascience.com/how-to-cross-validation-with-time-series-data-9802a06272c6/

4. GridSearchCV
Works the same as sklearn I think, but might need to be debugged.

Sources:
> https://www.kaggle.com/code/pythonafroz/step-by-step-guide-to-gridsearchcv



### Compare models
They differ, I'll write a more extensive decision, but I want to make sure my setup is correct. 



In [ ]:
data = Datasets('custom',split_method='sklearn')
data.load_custom_data(file_path='../data/')  


## Tasks
1. Load your Day 3 train/test splits.
2. Train your Day 4 winning model configuration on the **full training set**.
3. Evaluate on the **held-out test set** (never touched until now) with:
   accuracy, precision, recall, F1, and a confusion matrix. Given the class
   imbalance, report which metric you're treating as the "headline" number
   and justify why.
4. Plot an ROC curve and report AUC.
5. If your model type supports it, extract and plot feature importances (or
   coefficients) — which features actually drove predictions? Does it match
   your intuition from Day 3's EDA?
6. Save the trained model to disk (`joblib.dump`).
7. Write a **model card** — use the template below, filled out for real —
   and save it as `MODEL_CARD.md` in this model's folder.

In [ ]:
import sklearn.model_selection as model_selection
X = data.X
y = data.y
X_train, X_test, y_train, y_test = data.X_train, data.X_test, data.y_train, data.y_test
log_reg_params = {'C': 15, 'penalty': 'l1', 'solver': 'liblinear'}
knn_paeams = {'n_neighbors': 9, 'weights': 'uniform', 'metric': 'manhattan'}
model = KNeighborsClassifier(**knn_paeams)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

print("Classification Report for KNN:")
print(metrics.classification_report(y_test, y_pred))
print("Confusion Matrix for KNN:")
print(metrics.confusion_matrix(y_test, y_pred))


# cross-validation scores for the model
cv = StratifiedKfoldCV(n_samples=len(X), k=5, seed=42)
cv_sklearn = model_selection.StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = model_selection.cross_val_score(model, X, y, cv=cv_sklearn, scoring='roc_auc')
cv_means_log, cv_stds_log, cv_scores_log = cross_val_score(model, X, y, cv=cv)

print('-' * 30)
print("Mean scores with standard deviation for LogisticRegression:")
print("Accuracy: {:.4f} ± {:.4f}".format(cv_means_log['accuracy'], cv_stds_log['accuracy']))
print("Precision: {:.4f} ± {:.4f}".format(cv_means_log['precision'], cv_stds_log['precision']))
print("Recall: {:.4f} ± {:.4f}".format(cv_means_log['recall'], cv_stds_log['recall']))
print("F1 Score: {:.4f} ± {:.4f}".format(cv_means_log['f1'], cv_stds_log['f1']))
print("ROC-AUC: {:.4f} ± {:.4f}".format(cv_means_log['roc_auc'], cv_stds_log['roc_auc']))

print('-' * 30)
print('Sklearn functions')
print("Cross-validated ROC AUC:", np.round(cv_scores, 3))
print("Mean AUC:", round(cv_scores.mean(), 3))


features = data.X.columns
importances = model.feature_importances_ if hasattr(model, 'feature_importances_') else np.abs(model.coef_[0])
percent_importances = 100.0 * (importances / importances.sum())
print("Feature importances for Logistic Regression:")
for feature, importance in zip(features, percent_importances):
    print(f"{feature}: {importance:.2f}%")
metrics.RocCurveDisplay.from_estimator(model, X_test, y_test)
plt.title("ROC Curve for Logistic Regression")
plt.show()
